# Fine-tune `codelion/dhara-250m` on finance data (Google Colab)

Full-parameter SFT (250M params, no LoRA needed) on a mix of three finance datasets.

| Dataset | Format | What it teaches |
|---|---|---|
| `nvidia/Nemotron-SpecializedDomains-Finance-v1` | `messages` | Q&A grounded in SEC 10-K/10-Q excerpts |
| `lordtyrion/adaption-india-finance-qa-risks-v1` | `prompt` / `completion` / ... | Indian personal finance, GST/TDS, loans, scams |
| `ash0t/finance-slm-distillation-data` | `instruction` / `final_answer` / `think` | Quantitative word problems with worked solutions |

**Colab-specific behaviour**

- **Nemotron is downloaded, not streamed** — shard by shard, each shard filtered to short-enough rows and then **deleted**, with a disk budget (`NEMO_MAX_GB`) so you never fill Colab's disk.
- **Google Drive** holds checkpoints, the final model and the tokenised-data cache. If the runtime disconnects, re-run the notebook and training **resumes** from the last checkpoint; preprocessing is skipped.
- **Auto-config by GPU** (T4 / L4 / A100 / H100): precision, micro-batch size and gradient accumulation (effective batch stays constant).
- **Length-bucketed batching** (less padding, faster steps), pad-to-multiple-of-8, TF32 on Ampere+.
- **Optional time budget** (`TIME_BUDGET_MIN`) that saves and stops cleanly before Colab's session limit.

**Training design**: ChatML via `apply_chat_template`, completion-only loss, own cross-entropy (the model uses `trust_remote_code`), over-length examples dropped rather than truncated, reasoning traces stripped by default.

> **Caveat:** Dhara was trained for AR + block-diffusion + self-speculation. This notebook fine-tunes the **AR loss only**, so the other two modes may drift (last cell checks).

**Before running:** Runtime → Change runtime type → GPU (T4 works; L4/A100 is faster). Add an `HF_TOKEN` in Colab Secrets (key icon) for faster, rate-limit-free downloads.

## 0. Setup

In [ ]:
# Colab already ships torch / transformers / accelerate. Only add what is missing.
# hf_xet = fast downloads for Hugging Face's Xet-backed repos (the Nemotron set is one).
%pip install -q hf_xet datasets

## 1. Config

In [ ]:
MODEL_ID = "codelion/dhara-250m"

# ---- Colab storage
USE_DRIVE = True              # checkpoints + final model + data cache go to Google Drive (survives disconnects)
DRIVE_DIR = "MyDrive/dhara-finance"

# ---- datasets
NEMO_REPO  = "nvidia/Nemotron-SpecializedDomains-Finance-v1"
INDIA_REPO = "lordtyrion/adaption-india-finance-qa-risks-v1"
ASH_REPO   = "ash0t/finance-slm-distillation-data"

MAX_LEN        = 2048         # drop examples longer than this (model was trained to 4k)
NEMOTRON_N     = 20000        # stop downloading once this many usable Nemotron rows are collected
NEMO_MAX_GB    = 12           # hard disk budget for Nemotron shards (each is deleted after processing)
NEMO_MAX_SHARDS = None        # optional extra cap on number of shards to download
INDIA_N        = None         # None = use all (~35k)
ASH_N          = None         # None = use all (~14.7k)

INDIA_FIELD  = "completion"   # "completion" (short, structured) or "enhanced_completion" (long, markdown)
REASONING    = "strip"        # "strip" = answers only, "keep" = include <think> traces
ENGLISH_ONLY = True           # Dhara is English-only; India set contains some Bengali/Hindi rows
VAL_N        = 300

SYSTEM = ("You are an expert finance assistant. Answer accurately and concisely, show calculations "
          "step by step, state your assumptions, and flag risks or details the user should verify.")

# ---- training
EPOCHS        = 2
LR            = 3e-5
EFFECTIVE_BS  = 32            # sequences per optimizer step (kept constant across GPUs)
PER_DEVICE_BS = None          # None = auto from GPU memory; set an int to override
WEIGHT_DECAY  = 0.01
WARMUP_FRAC   = 0.03
GRAD_CKPT     = False         # custom-code model may not support it; try True only if you OOM
TIME_BUDGET_MIN = None        # e.g. 540 to stop cleanly after 9h; re-run the train cell to resume
SEED          = 42

# ---- generation for the before/after comparison
REP_PEN = 1.05                # model card says 1.15, but that penalises repeated digits in math answers

In [ ]:
import os, sys, math, random, json, re, time, hashlib, shutil, gzip
import numpy as np
import torch
import torch.nn.functional as F

os.environ["TOKENIZERS_PARALLELISM"] = "false"

IN_COLAB = "google.colab" in sys.modules
LOCAL_DIR = "/content" if IN_COLAB else "."

# ---- Drive
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = os.path.join("/content/drive", DRIVE_DIR)
else:
    WORK_DIR = os.path.join(LOCAL_DIR, "dhara-finance")
os.makedirs(WORK_DIR, exist_ok=True)
OUTPUT_DIR = os.path.join(WORK_DIR, "checkpoints")
FINAL_DIR  = os.path.join(WORK_DIR, "final")
NEMO_DIR   = os.path.join(LOCAL_DIR, "nemo_shards")   # always local disk (fast, temporary)
print("work dir:", WORK_DIR)

# ---- GPU
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU"
device = "cuda"
props = torch.cuda.get_device_properties(0)
GPU_GB = props.total_memory / 1e9
BF16 = torch.cuda.is_bf16_supported()
if props.major >= 8:                                   # Ampere+ (L4, A100, H100)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print(f"GPU: {props.name} | {GPU_GB:.0f} GB | bf16: {BF16}")

if PER_DEVICE_BS is None:
    PER_DEVICE_BS = 16 if GPU_GB >= 70 else 8 if GPU_GB >= 35 else 6 if GPU_GB >= 20 else 4
GRAD_ACCUM = max(1, math.ceil(EFFECTIVE_BS / PER_DEVICE_BS))
print(f"micro-batch {PER_DEVICE_BS} x grad-accum {GRAD_ACCUM} = {PER_DEVICE_BS * GRAD_ACCUM} sequences/step")

import transformers, datasets
print("transformers", transformers.__version__, "| datasets", datasets.__version__, "| torch", torch.__version__)

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
    print("Logged in with Colab secret HF_TOKEN")
except Exception:
    if os.environ.get("HF_TOKEN"):
        login(os.environ["HF_TOKEN"])
    else:
        print("No HF_TOKEN secret found. Downloads still work, but may be slower / rate-limited.")

## 2. Load tokenizer + model and sanity-check the forward pass

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
im_end_id = tok.convert_tokens_to_ids("<|im_end|>")
pad_id = tok.pad_token_id if tok.pad_token_id is not None else im_end_id
assert im_end_id is not None and im_end_id != tok.unk_token_id, "<|im_end|> not found in tokenizer"

# fp32 master weights; Trainer uses autocast (bf16 / fp16) for compute
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, trust_remote_code=True, torch_dtype=torch.float32).to(device)
print(f"params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

# Check 1: default forward is causal (needed for right-padding without an attention mask)
model.eval()
with torch.no_grad():
    x = tok("The quarterly report shows that revenue increased by ten percent compared to last year.",
            return_tensors="pt", add_special_tokens=False).input_ids.to(device)
    k = x.shape[1] // 2
    full = model(input_ids=x).logits
    part = model(input_ids=x[:, :k]).logits
    diff = (full[:, :k] - part).abs().max().item()
print(f"causality check, max logit diff on shared prefix: {diff:.2e}")
if diff > 1e-2:
    print("WARNING: forward pass does not look causal. Check the model's default attention mode before training.")

# Check 2: chat template
demo = tok.apply_chat_template(
    [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "What is TDS?"}],
    tokenize=False, add_generation_prompt=True)
print("\n--- chat template ---\n" + demo)

## 3. Data pipeline

Everything is normalised to `{"source", "user", "assistant"}`.

- **Nemotron** is downloaded **one shard at a time**: download, keep rows that can fit in `MAX_LEN`, delete the shard, repeat until `NEMOTRON_N` rows or the `NEMO_MAX_GB` budget is reached. Shard order is shuffled so you get a random sample of the dataset, not just the first file.
- **India QA**: its Hub viewer errors with a `string -> null` cast, so `load_dataset` may fail; the loader falls back to reading raw files.
- The next cell only **defines** functions. The cell after it runs them (or loads the cached result from Drive).

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict, load_from_disk
from huggingface_hub import HfApi, hf_hub_download, list_repo_files

THINK_RE = re.compile(r"<think>.*?</think>\s*", re.DOTALL)

def clean_answer(text):
    text = text or ""
    if REASONING == "strip":
        text = THINK_RE.sub("", text)
    return text.strip()

def with_think(think, answer):
    answer = (answer or "").strip()
    think = (think or "").strip()
    if REASONING == "keep" and think:
        return f"<think>\n{think}\n</think>\n\n{answer}"
    return answer

def mostly_latin(s, thr=0.9):
    letters = [c for c in s if c.isalpha()]
    if not letters:
        return True
    return sum(c.isascii() for c in letters) / len(letters) >= thr

def dedup(rows):
    seen, out = set(), []
    for r in rows:
        key = hash(r["user"])
        if key not in seen:
            seen.add(key)
            out.append(r)
    return out

def finalize(rows, name, cap):
    rows = [r for r in rows if r["user"].strip() and r["assistant"].strip()]
    if ENGLISH_ONLY:
        rows = [r for r in rows if mostly_latin(r["user"]) and mostly_latin(r["assistant"])]
    rows = dedup(rows)
    random.Random(SEED).shuffle(rows)
    if cap:
        rows = rows[:cap]
    print(f"{name}: {len(rows)} examples")
    return rows

# ---------------------------------------------------------------- generic loader (India, ash0t)
def load_rows(repo, cols, prefer=None, split="train"):
    try:
        ds = load_dataset(repo, split=split)
        keep = [c for c in cols if c in ds.column_names]
        return [dict(r) for r in ds.select_columns(keep)]
    except Exception as e:
        print(f"[{repo}] load_dataset failed ({type(e).__name__}); reading raw files instead.")
    import pandas as pd
    files = [f for f in list_repo_files(repo, repo_type="dataset")
             if f.lower().endswith((".parquet", ".jsonl", ".json", ".csv"))]
    if prefer:
        files = [f for f in files if f in prefer] or files
    for ext in (".parquet", ".jsonl", ".json", ".csv"):      # one file type only, avoids duplicate copies
        sel = [f for f in files if f.lower().endswith(ext)]
        if sel:
            files = sel
            break
    rows = []
    for f in files:
        p = hf_hub_download(repo, f, repo_type="dataset")
        fl = f.lower()
        if fl.endswith(".parquet"):
            df = pd.read_parquet(p)
            rows += df[[c for c in cols if c in df.columns]].to_dict("records")
        elif fl.endswith(".csv"):
            df = pd.read_csv(p)
            rows += df[[c for c in cols if c in df.columns]].to_dict("records")
        else:
            with open(p, encoding="utf-8") as fh:
                head = fh.read(1)
                fh.seek(0)
                data = json.load(fh) if head == "[" else (json.loads(l) for l in fh if l.strip())
                rows += [{c: r.get(c) for c in cols} for r in data]
    print(f"[{repo}] loaded {len(rows)} rows from {len(files)} file(s)")
    return rows

# ---------------------------------------------------------------- Nemotron: shard-by-shard download
def iter_messages(path):
    pl = path.lower()
    if pl.endswith(".parquet"):
        import pyarrow.parquet as pq
        pf = pq.ParquetFile(path)
        for batch in pf.iter_batches(columns=["messages"], batch_size=256):
            for m in batch.column("messages").to_pylist():
                yield m
    elif pl.endswith(".json"):
        with open(path, encoding="utf-8") as fh:
            for ex in json.load(fh):
                yield ex["messages"]
    else:
        opener = gzip.open if pl.endswith(".gz") else open
        with opener(path, "rt", encoding="utf-8") as fh:
            for line in fh:
                if line.strip():
                    yield json.loads(line)["messages"]

def get_nemotron():
    api = HfApi()
    entries = [e for e in api.list_repo_tree(NEMO_REPO, repo_type="dataset", recursive=True)
               if hasattr(e, "size") and re.search(r"\.(parquet|jsonl|json)(\.gz)?$", e.path)
               and os.path.basename(e.path) not in ("dataset_infos.json", "config.json")]
    for ext in (".parquet", ".jsonl", ".json"):               # use one format only
        sel = [e for e in entries if re.search(re.escape(ext) + r"(\.gz)?$", e.path)]
        if sel:
            entries = sel
            break
    if not entries:
        print("No data files found in the Nemotron repo listing; falling back to streaming.")
        return get_nemotron_stream()

    total_gb = sum(e.size for e in entries) / 1e9
    print(f"nemotron: {len(entries)} shard(s), {total_gb:.1f} GB total; will download until "
          f"{NEMOTRON_N} usable rows or {NEMO_MAX_GB} GB")
    random.Random(SEED).shuffle(entries)
    os.makedirs(NEMO_DIR, exist_ok=True)

    max_chars = int(MAX_LEN * 4)          # rough chars/token pre-filter; the exact token check comes later
    rows, used_gb, n_shards, scanned = [], 0.0, 0, 0
    for e in entries:
        if NEMOTRON_N and len(rows) >= NEMOTRON_N:
            break
        if NEMO_MAX_SHARDS and n_shards >= NEMO_MAX_SHARDS:
            break
        gb = e.size / 1e9
        if used_gb + gb > NEMO_MAX_GB:
            print(f"  skip {e.path} ({gb:.1f} GB) - over the disk budget")
            continue
        t0 = time.time()
        p = hf_hub_download(NEMO_REPO, e.path, repo_type="dataset", local_dir=NEMO_DIR)
        used_gb += gb; n_shards += 1
        before = len(rows)
        for msgs in iter_messages(p):
            scanned += 1
            if isinstance(msgs, str):
                msgs = json.loads(msgs)
            users = [m["content"] for m in msgs if m["role"] == "user"]
            assts = [m["content"] for m in msgs if m["role"] == "assistant"]
            if users and assts and len(users[-1]) + len(assts[-1]) <= max_chars:
                rows.append({"source": "nemotron", "user": users[-1], "assistant": clean_answer(assts[-1])})
        os.remove(p)                                           # free disk right away
        print(f"  shard {n_shards}: {gb:.2f} GB in {time.time()-t0:.0f}s, +{len(rows)-before} usable rows "
              f"(total {len(rows)}, scanned {scanned})")
    shutil.rmtree(NEMO_DIR, ignore_errors=True)
    return rows

def get_nemotron_stream():
    ds = load_dataset(NEMO_REPO, split="train", streaming=True).shuffle(seed=SEED, buffer_size=1000)
    rows, scanned, max_chars = [], 0, int(MAX_LEN * 4)
    for ex in ds:
        scanned += 1
        msgs = ex["messages"]
        if isinstance(msgs, str):
            msgs = json.loads(msgs)
        users = [m["content"] for m in msgs if m["role"] == "user"]
        assts = [m["content"] for m in msgs if m["role"] == "assistant"]
        if users and assts and len(users[-1]) + len(assts[-1]) <= max_chars:
            rows.append({"source": "nemotron", "user": users[-1], "assistant": clean_answer(assts[-1])})
        if (NEMOTRON_N and len(rows) >= NEMOTRON_N) or scanned >= 40000:
            break
    return rows

# ---------------------------------------------------------------- tokenisation with completion-only labels
def build_ids(batch_rows):
    prompts = [tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM}, {"role": "user", "content": r["user"]}],
        tokenize=False, add_generation_prompt=True) for r in batch_rows]
    comps = [r["assistant"].strip() + "<|im_end|>" for r in batch_rows]
    p_ids = tok(prompts, add_special_tokens=False).input_ids
    c_ids = tok(comps, add_special_tokens=False).input_ids
    out = []
    for r, p, c in zip(batch_rows, p_ids, c_ids):
        if len(p) + len(c) > MAX_LEN:        # drop, never truncate (truncation would remove <|im_end|>)
            continue
        out.append({"source": r["source"], "input_ids": p + c,
                    "labels": [-100] * len(p) + c, "n_comp": len(c)})
    return out

def build_datasets():
    # 3a Nemotron
    nemo = finalize(get_nemotron(), "nemotron", NEMOTRON_N)
    # 3b India
    india = []
    for r in load_rows(INDIA_REPO, ["prompt", "completion", "enhanced_completion", "reasoning_trace"]):
        ans = r.get(INDIA_FIELD) or r.get("completion") or ""
        india.append({"source": "india", "user": (r.get("prompt") or "").strip(),
                      "assistant": with_think(r.get("reasoning_trace"), ans)})
    india = finalize(india, "india", INDIA_N)
    # 3c ash0t
    ash = [{"source": "ash0t", "user": (r.get("instruction") or "").strip(),
            "assistant": with_think(r.get("think"), r.get("final_answer"))}
           for r in load_rows(ASH_REPO, ["instruction", "final_answer", "think"],
                              prefer=["df_train.parquet"], split="train")]
    ash = finalize(ash, "ash0t", ASH_N)

    # eyeball one example per source
    for name, rows in [("nemotron", nemo), ("india", india), ("ash0t", ash)]:
        if not rows:
            print(f"!! {name} is EMPTY"); continue
        print(f"\n===== {name} =====")
        print("USER     :", rows[0]["user"][:400].replace("\n", " "), "...")
        print("ASSISTANT:", rows[0]["assistant"][:400].replace("\n", " "), "...")

    all_rows = nemo + india + ash
    random.Random(SEED).shuffle(all_rows)
    encoded = []
    for i in range(0, len(all_rows), 1000):
        encoded += build_ids(all_rows[i:i + 1000])

    print(f"\nkept {len(encoded)} / {len(all_rows)} examples after the {MAX_LEN}-token filter")
    for s in ("nemotron", "india", "ash0t"):
        sub = [e for e in encoded if e["source"] == s]
        if sub:
            tot = np.array([len(e["input_ids"]) for e in sub]); comp = np.array([e["n_comp"] for e in sub])
            print(f"  {s:9s} n={len(sub):6d}  mean_len={tot.mean():7.0f}  mean_answer_tokens={comp.mean():6.0f}")

    keep = lambda e: {"input_ids": e["input_ids"], "labels": e["labels"]}
    return DatasetDict({
        "validation": Dataset.from_list([keep(e) for e in encoded[:VAL_N]]),
        "train": Dataset.from_list([keep(e) for e in encoded[VAL_N:]]),
    })

### Build, or load from cache

The cache key is a hash of every setting that affects the data. Change `MAX_LEN`, `REASONING`, dataset caps etc. and it rebuilds; otherwise a reconnect skips the whole download + tokenisation.

In [ ]:
cfg = dict(model=MODEL_ID, max_len=MAX_LEN, nemo_n=NEMOTRON_N, nemo_gb=NEMO_MAX_GB, nemo_shards=NEMO_MAX_SHARDS,
           india_n=INDIA_N, ash_n=ASH_N, india_field=INDIA_FIELD, reasoning=REASONING,
           english=ENGLISH_ONLY, val=VAL_N, system=SYSTEM, seed=SEED)
key = hashlib.md5(json.dumps(cfg, sort_keys=True).encode()).hexdigest()[:10]
DATA_CACHE = os.path.join(WORK_DIR, f"data_{key}")

if os.path.exists(os.path.join(DATA_CACHE, "dataset_dict.json")):
    print("Loading cached tokenised data from", DATA_CACHE)
    dsd = load_from_disk(DATA_CACHE)
else:
    dsd = build_datasets()
    dsd.save_to_disk(DATA_CACHE)
    print("\nCached to", DATA_CACHE)

train_ds, val_ds = dsd["train"], dsd["validation"]
n_ans_tokens = sum(sum(1 for t in l if t != -100) for l in train_ds["labels"])
print(f"\ntrain: {len(train_ds)} | val: {len(val_ds)} | trainable answer tokens/epoch: {n_ans_tokens/1e6:.1f}M")

# confirm the masking looks right
ex = train_ds[0]
print("\n--- masked prompt (not trained on) ---")
print(tok.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l == -100])[-350:])
print("\n--- trained answer ---")
print(tok.decode([l for l in ex["labels"] if l != -100])[:500])

## 4. Train

Colab-oriented pieces in this cell:
- **`LengthBucketSampler`** shuffles, sorts inside mega-batches by length, then shuffles the batches. Similar-length sequences share a batch, so much less compute is wasted on padding.
- **`TimeBudget`** callback saves and stops cleanly if `TIME_BUDGET_MIN` is set.
- Checkpoints go to Drive (about 3 GB each: weights + Adam state), and only the latest is kept.

In [ ]:
from torch.utils.data import Sampler
from transformers import Trainer, TrainingArguments, TrainerCallback
from transformers.trainer_utils import get_last_checkpoint

class Collator:
    def __init__(self, pad_id, multiple=8):
        self.pad_id, self.multiple = pad_id, multiple
    def __call__(self, batch):
        L = max(len(b["input_ids"]) for b in batch)
        L = ((L + self.multiple - 1) // self.multiple) * self.multiple      # tensor-core friendly
        ids = torch.full((len(batch), L), self.pad_id, dtype=torch.long)
        lab = torch.full((len(batch), L), -100, dtype=torch.long)
        for i, b in enumerate(batch):
            n = len(b["input_ids"])
            ids[i, :n] = torch.tensor(b["input_ids"])
            lab[i, :n] = torch.tensor(b["labels"])
        return {"input_ids": ids, "labels": lab}

class LengthBucketSampler(Sampler):
    def __init__(self, lengths, batch_size, mega=64, seed=0):
        self.lengths, self.bs, self.mega, self.seed, self.epoch = lengths, batch_size, mega, seed, 0
    def __len__(self):
        return len(self.lengths)
    def __iter__(self):
        g = torch.Generator(); g.manual_seed(self.seed + self.epoch); self.epoch += 1
        idx = torch.randperm(len(self.lengths), generator=g).tolist()
        chunk = self.bs * self.mega
        full, partial = [], []
        for i in range(0, len(idx), chunk):
            c = sorted(idx[i:i + chunk], key=lambda j: -self.lengths[j])
            for k in range(0, len(c), self.bs):
                b = c[k:k + self.bs]
                (full if len(b) == self.bs else partial).append(b)
        order = torch.randperm(len(full), generator=g).tolist()
        for o in order:
            yield from full[o]
        for b in partial:                       # a short batch, if any, goes last so alignment is preserved
            yield from b

class SFTTrainer(Trainer):
    # Own cross-entropy: no reliance on the custom model's labels= handling.
    # Only answer positions are cast to fp32, which saves a lot of memory.
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out = model(input_ids=inputs["input_ids"])
        logits = out.logits if hasattr(out, "logits") else out[0]
        shift_logits, shift_labels = logits[:, :-1, :], labels[:, 1:]
        mask = shift_labels != -100
        loss = F.cross_entropy(shift_logits[mask].float(), shift_labels[mask])
        return (loss, out) if return_outputs else loss

    def _get_train_sampler(self, *args, **kwargs):
        return LengthBucketSampler(self.train_lengths, self.args.per_device_train_batch_size, seed=self.args.seed)

class TimeBudget(TrainerCallback):
    def __init__(self, minutes):
        self.limit, self.t0 = minutes * 60, None
    def on_train_begin(self, args, state, control, **kw):
        self.t0 = time.time()
    def on_step_end(self, args, state, control, **kw):
        if time.time() - self.t0 > self.limit:
            print(f"\nTime budget reached at step {state.global_step}: saving and stopping. Re-run this cell to resume.")
            control.should_save = True
            control.should_training_stop = True
        return control

steps_per_epoch = math.ceil(len(train_ds) / (PER_DEVICE_BS * GRAD_ACCUM))
total_steps = steps_per_epoch * EPOCHS
ckpt_every = max(50, steps_per_epoch // 2)          # every half epoch: a Drive write is ~3 GB
print(f"steps/epoch: {steps_per_epoch} | total: {total_steps} | eval+checkpoint every {ckpt_every}")

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_BS,
    per_device_eval_batch_size=PER_DEVICE_BS * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=int(WARMUP_FRAC * total_steps),
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=1.0,
    bf16=BF16,
    fp16=not BF16,
    gradient_checkpointing=GRAD_CKPT,
    optim="adamw_torch_fused",
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=ckpt_every,
    save_strategy="steps",
    save_steps=ckpt_every,
    save_total_limit=1,
    remove_unused_columns=False,
    dataloader_num_workers=2,
    report_to="none",
    seed=SEED,
)

callbacks = [TimeBudget(TIME_BUDGET_MIN)] if TIME_BUDGET_MIN else []
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                     data_collator=Collator(pad_id), callbacks=callbacks)
trainer.train_lengths = [len(x) for x in train_ds["input_ids"]]

last_ckpt = get_last_checkpoint(OUTPUT_DIR) if os.path.isdir(OUTPUT_DIR) else None
if last_ckpt is None:
    base_eval = trainer.evaluate()
    print(f"BASE val loss: {base_eval['eval_loss']:.4f} | ppl: {math.exp(base_eval['eval_loss']):.2f}")
else:
    base_eval = None
    print("Found checkpoint, will resume from:", last_ckpt)

In [ ]:
trainer.train(resume_from_checkpoint=last_ckpt)

In [ ]:
final_eval = trainer.evaluate()
if base_eval:
    print(f"BASE  val loss: {base_eval['eval_loss']:.4f} | ppl: {math.exp(base_eval['eval_loss']):.2f}")
print(f"TUNED val loss: {final_eval['eval_loss']:.4f} | ppl: {math.exp(final_eval['eval_loss']):.2f}")

## 5. Before / after generation

Same prompts, base vs fine-tuned, AR mode. Prompt 1 is checkable by hand: taxable value Rs 1,000, GST Rs 180.

In [ ]:
@torch.no_grad()
def chat(m, question, max_new_tokens=350, temperature=0.7, top_p=0.9):
    m.eval()
    prompt = tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}],
        tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt", add_special_tokens=False).input_ids.to(m.device)
    out = m.generate(ids, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature,
                     top_p=top_p, repetition_penalty=REP_PEN, eos_token_id=im_end_id, pad_token_id=pad_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

test_prompts = [
    "A customer pays Rs 1,180 for an item whose price includes 18% GST. What are the taxable value and GST amount?",
    "The bank sent a message saying 'TDS has been deducted' on my FD interest. Have I lost the money?",
    "A project costs $250,000 and returns $61,900 a year for 5 years. At a 12% cost of capital, what is the present value of the inflows?",
    "What does the risk factors section of a 10-K typically disclose, and why does it matter to investors?",
    "An agent says he will return part of his commission if I buy a policy today. Is that a good deal?",
]

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, trust_remote_code=True, torch_dtype=torch.float32).to(device)
tuned = trainer.model

for q in test_prompts:
    print("=" * 100)
    print("Q:", q)
    print("\n[BASE]\n", chat(base, q))
    print("\n[TUNED]\n", chat(tuned, q))
del base
torch.cuda.empty_cache()

## 6. Save to Drive (and optionally push to the Hub)

In [ ]:
HUB_REPO = None     # e.g. "your-username/dhara-250m-finance"

trainer.model.save_pretrained(FINAL_DIR, safe_serialization=True)
tok.save_pretrained(FINAL_DIR)

# trust_remote_code models need their .py files next to the weights; copy them if save_pretrained skipped them
if not any(f.endswith(".py") for f in os.listdir(FINAL_DIR)):
    from huggingface_hub import snapshot_download
    src = snapshot_download(MODEL_ID, allow_patterns=["*.py"])
    for f in os.listdir(src):
        if f.endswith(".py"):
            shutil.copy(os.path.join(src, f), FINAL_DIR)
print("saved to", FINAL_DIR, "->", sorted(os.listdir(FINAL_DIR)))

if HUB_REPO:
    api = HfApi()
    api.create_repo(HUB_REPO, exist_ok=True)
    api.upload_folder(folder_path=FINAL_DIR, repo_id=HUB_REPO, commit_message="Dhara-250M finance SFT")
    print("pushed to", f"https://huggingface.co/{HUB_REPO}")

In [ ]:
# Reload from disk to prove the saved model works
reloaded = AutoModelForCausalLM.from_pretrained(FINAL_DIR, trust_remote_code=True, torch_dtype=torch.float32).to(device)
print(chat(reloaded, "Explain what a lien and a hold on a bank account mean, in simple words."))

## 7. (Optional) Check the other decoding modes

Only the AR loss was used, so block-diffusion and self-speculation were not trained on this data. If they matter to you, compare here; if they degrade, serve in AR mode, or add a joint AR + masked-diffusion loss following NVIDIA's Tri-Mode recipe.

In [ ]:
q = "A customer pays Rs 1,180 for an item whose price includes 18% GST. What are the taxable value and GST amount?"
prompt = tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}],
                                 tokenize=False, add_generation_prompt=True)
ids = tok(prompt, return_tensors="pt", add_special_tokens=False).input_ids.to(device)
m = trainer.model.eval()

for name, fn in [
    ("block-diffusion", lambda: m.generate_diffusion(ids, block_len=32, threshold=0.5, max_new_tokens=200)),
    ("self-speculation", lambda: m.generate_self_spec(ids, k=8, max_new_tokens=200)),
]:
    try:
        with torch.no_grad():
            out = fn()
        print(f"--- {name} ---\n{tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()}\n")
    except Exception as e:
        print(f"--- {name} failed: {type(e).__name__}: {e}\n")